In [30]:
# 1-1
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/8_1_1.csv')
# print(df.info())
# print(df.head())
print(df.groupby(['대륙'])['맥주소비량'].mean().sort_values(ascending=False).idxmax())
print(df[df['대륙']=='SA'].groupby('국가')['맥주소비량'].sum().sort_values(ascending=False).index[4])
print(np.round(df[df['국가']=='Venezuela']['맥주소비량'].mean(),0))

SA
Venezuela
253.0


In [62]:
# 1-2
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/8_1_2.csv')
# print(df.info())
# print(df.head())
df1=df.groupby('국가')[['관광', '사무', '공무', '유학', '기타']].sum()
df1['합계']=df1[['관광', '사무', '공무', '유학', '기타']].sum(axis=1)
df1['관광객비율']=np.round((df1['관광']/df1['합계']),3)
# print(df1.info())
# print(df1.head())
print(df1[df1.index==df1['관광객비율'].sort_values(ascending=False).index[1]]['관광'].values[0])
print(np.round(df[df['국가']==df1['관광'].sort_values(ascending=False).index[1]]['공무'].mean(),0))
a=df1[df1.index==df1['관광객비율'].sort_values(ascending=False).index[1]]['관광']
b=np.round(df[df['국가']==df1['관광'].sort_values(ascending=False).index[1]]['공무'].mean(),0)
print(a.values[0]+b)

9039
494.0
9533.0


In [69]:
# 1-3
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/8_1_3.csv')
# print(df.info())
# print(df.head())
from sklearn.preprocessing import minmax_scale
df[['CO(GT)', 'NMHC(GT)']]=minmax_scale(df[['CO(GT)', 'NMHC(GT)']])
# print(df.info())
# print(df.head())
print(np.round(df['CO(GT)'].std(),2))
print(np.round(df['NMHC(GT)'].std(), 2))

0.37
0.15


In [27]:
# 2
import numpy as np
import pandas as pd
train_df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/8_2_train.csv')
test_df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/8_2_test.csv')
# print(train_df.info())
# print(train_df.head())
# print(test_df.info())

train_y=train_df['count']
train_X=train_df.drop(columns=['ID', 'count'])
test_X=test_df.drop(columns=['ID', 'count'])
# print(train_X.info())
# print(train_y.info())
# print(test_X.info())

from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import get_scorer_names, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder

train_X, valid_X, train_y, valid_y=train_test_split(train_X, train_y, test_size=0.3)
# print(train_X.shape, train_y.shape, valid_X.shape, valid_y.shape)

obj=train_X.select_dtypes('object').columns.tolist()
num=train_X.select_dtypes('number').columns.tolist()

encoder=OneHotEncoder(sparse_output=False, handle_unknown='ignore')
train_X_encoder=encoder.fit_transform(train_X[obj])
valid_X_encoder=encoder.transform(valid_X[obj])
test_X_encoder=encoder.transform(test_X[obj])

train_X=np.concatenate([train_X_encoder, train_X[num]],axis=1)
valid_X=np.concatenate([valid_X_encoder, valid_X[num]],axis=1)
test_X=np.concatenate([test_X_encoder, test_X[num]],axis=1)

rf=RandomForestRegressor()
rf.fit(train_X, train_y)
pred1=rf.predict(valid_X)
print(valid_y.mean())
print(mean_absolute_error(valid_y, pred1))

params={
    'max_depth': [10,20,30],
    'min_samples_split': [2,5,10]
}
cv=GridSearchCV(estimator=rf, param_grid=params, scoring='neg_mean_absolute_error', cv=3)
cv.fit(train_X, train_y)
pred2=cv.predict(valid_X)
print(mean_absolute_error(valid_y, pred2))

print((mean_absolute_error(valid_y, pred2)/valid_y.mean())*100)

pred3=cv.predict(test_X)

pd.DataFrame(pred3, columns=['pred']).to_csv('result.csv', index=False)

184.92105263157896
130.11464619883043
121.0082554828931
65.43779291802957


In [3]:
# 3-1
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/8_3_1.csv')
# print(df.info())
# print(df.head())

print(np.round(df['before'].mean(),2))
print(np.round(df['before'].std(),2))
print(np.round(df['after'].mean(),2))
print(np.round(df['after'].std(),2))
from scipy.stats import wilcoxon
print(np.round(wilcoxon(df['before'], df['after']).statistic, 2))
print(wilcoxon(df['before'], df['after']).pvalue < 0.05)

8.21
1.71
7.23
1.96
72.0
True


In [13]:
# 3-2
import numpy as np
import pandas as pd
train_df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/8_3_2_train.csv')
test_df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/8_3_2_test.csv')
# print(train_df.info())
# print(train_df.head())
# print(test_df.info())

import statsmodels.api as sm
import statsmodels.formula.api as smf
model=smf.ols('productivity ~ hours + age + experience', data=train_df).fit()
# print(model.summary())
print(model.params[1:].idxmax())
print((model.pvalues[1:] > 0.05).count())
test_y=test_df['productivity']
test_X=test_df.drop(columns=['productivity'])
pred1=model.predict(test_X)
from sklearn.metrics import r2_score
print(np.round(r2_score(test_y, pred1)*100,0))

hours
3
80.0
